In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive
import time
from tinyconformal.series import ConformalDistributionTimeSeriesRegressor
from tinyshift.modelling import DMSTLWrapper, fourier_seasonality
from utilsforecast.preprocessing import fill_gaps
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from mlforecast import MLForecast
from statsforecast.models import SeasonalNaive, AutoETS

# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
df = fourier_seasonality(df, "ds", seasonality=["monthly"])
days_obsoletes=180

/home/heylucasleao/tinyconformal/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
horizon = 12
train = df[:-horizon]
test = df[-horizon:]
models = [LinearRegression(), RandomForestRegressor(n_estimators=100, random_state=42)]

In [3]:
def seasonal_callable(period):
    return AutoETS(
        season_length=period,
        model="ZNA",
        alias=f"AutoETS-{period}",
    )


def trend_model_callable():
    return AutoETS(
        model="ZAN",
        alias="Trend-AutoETS",
    )


def residual_model_callable(nlags, freq):
    return MLForecast(
        models=models,
        lags=nlags,
        freq=freq,
    )


dmstl = DMSTLWrapper(
    residual_model_callable=residual_model_callable,
    freq="MS",
    trend_model_callable=trend_model_callable,
    seasonal_model_callable=seasonal_callable,
    pami_params={"fallback": 1},
    mode="local"
)

In [25]:

model_nixtla = ConformalDistributionTimeSeriesRegressor(
    learner=dmstl,
    horizon=7,
    n_windows=15,
    alpha=0.10,
    target_col="y",
    id_col="unique_id",
    time_col="ds",
)

start_time = time.perf_counter()
model_nixtla.fit(
    df=train,
    static_features=[]
)

,learner,DMSTLWrapper(...79ecd80fc700>)
,horizon,7
,n_windows,15
,alpha,0.1
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'
,mode,'local'
,residual_model_callable,<function res...x79ecd7f6d870>
,freq,'MS'
,season_length,'auto'


In [26]:
model_nixtla.evaluate(test, h=7)

,model,level,alpha,coverage_rate,interval_width_mean,mwis,mae,mbe,mse
0,LinearRegression,90%,0.1,0.857,95.316,100.434,15.143,5.549,377.574
1,RandomForestRegressor,90%,0.1,0.857,94.004,115.046,14.249,7.598,383.710
